# IS 6482 - Week 8 — Recommender Systems

**Author:** Varun Gupta

**Date:** 29 March, 2026

We will use the [MovieLens 100K dataset](https://grouplens.org/datasets/movielens/100k/) and the
[Surprise library](https://surpriselib.com/) (Simple Python RecommendatIon System Engine) to compare collaborative filtering and Matrix Factorization based recommender systems

- EDA -- what are the most highly rated movies?
- user-user similarity based recommendations for a target user
- item-item similarity based recommendations for a target user
- matrix factorization based recommendations for a target user
- How do these 4 sets of recommendations compare?
- Visualize the top movie recommendations for the target user using two latend factors

## Install packages (Colab)

In [ ]:
# This needs to be done before loading numpy
# Temporarily downgrade numpy for compatibility with surprise
!pip install "numpy<2" --force-reinstall

In [ ]:
!pip install -q scikit-surprise

## Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from surprise import Dataset, Reader, KNNBasic, SVD
from surprise.model_selection import train_test_split
from surprise import accuracy
from surprise.accuracy import rmse
from surprise.model_selection import cross_validate

from google.colab import drive

pd.set_option('display.max_columns', 50)
RANDOM_STATE = 42

## Load MovieLens 100K in Surprise

In [ ]:
# Also make a DataFrame of ratings for summaries
ratings = pd.read_csv('/content/ml-100k-ratings.csv')
ratings["rating"] = ratings["rating"].astype(float)
ratings["user"] = ratings["user"].astype(str)
ratings["item"] = ratings["item"].astype(str)

display(ratings.head())

movie_titles = pd.read_csv('/content/ml-100k-movie_titles.csv')
movie_titles["item"] = movie_titles["item"].astype(str)
display(movie_titles.head())

# We will be producing top recommendations for this user id
target_user = "196"
example_item = "408"

In [ ]:
# Create a reader and read data into the internal format of Surprise
reader = Reader(rating_scale=(1, 5))
data = Dataset.load_from_df(ratings[["user", "item", "rating"]], reader)
# Train test split (the next function is Surprise's train_test_split, not sklearn's)
trainset, testset = train_test_split(data, test_size=0.25, shuffle=True, random_state=RANDOM_STATE)

## Part 1 — Popularity baseline

In [ ]:
# Top movies based on baseline popularity
pop = (
    ratings.groupby("item")
    .agg(mean_rating=("rating", "mean"), n_ratings=("rating", "size"))
    .query("n_ratings >= 50")
    .sort_values(["mean_rating", "n_ratings"], ascending=False)
    .reset_index()
)

pop_with_titles = pop.merge(movie_titles, on="item", how="left")
display(pop_with_titles[["item", "title", "mean_rating", "n_ratings"]].head(10))

In [ ]:
# Top 10 movies rated by target user
target_user_ratings = ratings[ratings["user"] == target_user].sort_values("rating", ascending=False)
target_user_ratings = target_user_ratings.merge(movie_titles, on="item", how="left")
target_user_ratings.head(10)

## Define helpers

In [ ]:
# Helper functions

# This function takes a user_id, the trained scoring algorithm algo, and the training data trainset
# It then looks at all items that were not rated by user_id in train_set
# and uses algo.predict to prect the score
# All of this data (a list of tuples) is then passed to top_n_predictions
# which sorts by estimated score (est) and returns the top n as a DataFrame
def recommend_for_user(algo, trainset, user_id, n=10):
    seen = {trainset.to_raw_iid(iid) for (iid, _) in trainset.ur[trainset.to_inner_uid(user_id)]}
    all_items = list(trainset._raw2inner_id_items.keys())
    candidates = [iid for iid in all_items if iid not in seen]
    preds = [algo.predict(user_id, iid) for iid in candidates]
    return top_n_from_predictions(preds, n=n)

def top_n_from_predictions(predictions, n=10):
    rows = [(p.uid, p.iid, p.est) for p in predictions]
    out = pd.DataFrame(rows, columns=["user", "item", "score"])
    return out.sort_values("score", ascending=False).head(n)


## Part 2 — user-user and item-item collaborative filtering

### User-user cosine similarity based CF

In [ ]:
# Use KNN with neighborhood side of 40 and cosine similarity
# Surprise's KNN uses similarity weighted voting
user_user = KNNBasic(k= 40, sim_options={"name": "cosine", "user_based": True}, verbose=False)
user_user.fit(trainset)

# get a prediction for specific users and items.
pred = user_user.predict('196', '408', verbose=True)

In [ ]:
# Top 10 recommendation based user-user CF
uu_recs = recommend_for_user(user_user, trainset, target_user, n=10).merge(movie_titles, on="item", how="left")
print("User-user CF recommendations")
display(uu_recs[["item", "title", "score"]])

### Item-item cosine similarity based CF

In [ ]:
# KNN with Item-item cosine similarity
item_item = KNNBasic(k=40, sim_options={"name": "cosine", "user_based": False}, verbose=False)
item_item.fit(trainset)

# Make prediction for one user-item pair
pred = item_item.predict('196', '408', verbose=True)

In [ ]:
# Top 10 recommendations
ii_recs = recommend_for_user(item_item, trainset, target_user, n=10).merge(movie_titles, on="item", how="left")
print("Item-item CF recommendations")
display(ii_recs[["item", "title", "score"]])

## Part 3 — Compare the recommended top 10 lists for the baseline and collaborative filtering approaches

In [ ]:
pop_list = pop_with_titles[["item", "title", "mean_rating"]].head(10).rename(columns={"mean_rating": "score"})
pop_list["model"] = "Popularity"

uu_list = uu_recs[["item", "title", "score"]].copy()
uu_list["model"] = "User-user CF"

ii_list = ii_recs[["item", "title", "score"]].copy()
ii_list["model"] = "Item-item CF"

comparison = pd.concat([pop_list, uu_list, ii_list], ignore_index=True)
display(comparison)

## Part 4 — matrix factorization

In [ ]:
# learn latent representations of users and movies (after subtracting baselines)
# here we are asking for a fit with 20 factors
svd = SVD(n_factors=20, n_epochs=20, random_state=RANDOM_STATE)
svd.fit(trainset)

mf_recs = recommend_for_user(svd, trainset, target_user, n=10).merge(movie_titles, on="item", how="left")
display(mf_recs[["item", "title", "score"]])

In [ ]:
# Show all three lists side by side for the same target user
summary = pd.DataFrame({
    "Popularity": pop_with_titles["title"].head(10).tolist(),
    "User-user CF": uu_recs["title"].head(10).tolist(),
    "Item-item CF": ii_recs["title"].head(10).tolist(),
    "Matrix factorization": mf_recs["title"].head(10).tolist(),
})
display(summary)

### Inspect one matrix-factorization prediction

Surprise's SVD model predicts:
$
\hat r_{ui} = \mu + b_u + b_i + p_u^T q_i
$

The next cell extracts the pieces for one recommended movie.

In [ ]:
# Find the "inner" id used by Surprise for the "raw" user and item its
u = trainset.to_inner_uid(target_user)
i = trainset.to_inner_iid(example_item)

movie_name = movie_titles.loc[movie_titles["item"] == example_item, "title"].values[0]

print(f"User {target_user}'s latent factor: ")
print(np.round(svd.pu[u],3))
print('-'*50)
print(f"Item {example_item} ({movie_name}) latent factor: ")
print(np.round(svd.qi[i],3))
print('-'*50)
pieces = pd.Series({
    "global_mean": trainset.global_mean,
    "user_bias": svd.bu[u],
    "item_bias": svd.bi[i],
    "interaction": np.dot(svd.pu[u], svd.qi[i]),
})

pieces["predicted_rating"] = pieces.sum()
print(pieces.round(3))


## Part 5 - Visualize the top recommendations using first two latent factors

In [ ]:
# For visualization only you do not need to worry about this code
# Do not worry about this code

summary = pd.DataFrame({
    "Popularity": pop_with_titles["title"].head(10).tolist(),
    "User-user CF": uu_recs["title"].head(10).tolist(),
    "Item-item CF": ii_recs["title"].head(10).tolist(),
    "Matrix factorization": mf_recs["title"].head(10).tolist(),
})

top_movies = pd.concat([pop_with_titles['item'].head(10), uu_recs['item'].head(10), ii_recs['item'].head(10), mf_recs['item'].head(10)], ignore_index=True)
movie_ids = pd.unique(top_movies)

movie_latent = pd.DataFrame({
    "item": movie_ids,
    "latent_1": svd.qi[list(map(trainset.to_inner_iid, movie_ids)), 0],
    "latent_2": svd.qi[list(map(trainset.to_inner_iid, movie_ids)), 1]
})

movie_latent["item"] = movie_latent["item"].astype(str) # Keep as string to match movie_titles 'item' type

plot_df = movie_latent.merge(movie_titles, on="item", how="left")

fig, ax = plt.subplots(figsize=(10, 8), dpi=250)

ax.scatter(
    plot_df["latent_1"],
    plot_df["latent_2"],
    s=120,
    edgecolors="black"
)

for _, row in plot_df.iterrows():
    ax.text(
        row["latent_1"] + 0.01,
        row["latent_2"] + 0.01,
        row["title"],
        fontsize=10
    )

ax.axhline(0, color="gray", linewidth=1)
ax.axvline(0, color="gray", linewidth=1)
ax.set_title("Recommended movies in latent space", fontsize=18)
ax.set_xlabel("Latent factor 1", fontsize=14)
ax.set_ylabel("Latent factor 2", fontsize=14)
ax.tick_params(labelsize=11)

plt.tight_layout()
plt.show()

## Exercise
We did not do cross-validation for number of factors and number of neighbors. Look at the documentation of Surprise to find best k and best n_factors using cross validation